In [1]:
!pip install opencv-python numpy

In [2]:
pip install pillow arabic-reshaper python-bidi

Note: you may need to restart the kernel to use updated packages.


In [5]:


import os
import cv2
import numpy as np

# ============ عدّل هذين المسارين حسب جهازك ============
video_path = r"D:\Downloads eadge\انشئ_فيديو_تكون_الخلفيه_معزوله.mp4"
output_path = r"D:\coop traning\weak3\processed_output.mp4"
# ======================================================


MIN_AREA_RATIO = 0.015


PAD_LEFT = 0.12
PAD_RIGHT = 0.12
PAD_TOP = 0.20
PAD_BOTTOM = 0.35


def compute_focus_mask(frame):
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    lap_abs = np.abs(lap)

    energy = cv2.boxFilter(lap_abs, -1, (21, 21))
    energy_norm = cv2.normalize(energy, None, 0, 255, cv2.NORM_MINMAX).astype("uint8")
    _, mask = cv2.threshold(energy_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (61, 81))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    kernel_open = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    return mask


def find_main_object(mask, frame_area):
    """يرجع صندوق أكبر منطقة حادة إذا كانت مساحتها كافية، وإلا None."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest) < frame_area * MIN_AREA_RATIO:
        return None
    return cv2.boundingRect(biggest)


def pad_box(box, frame_shape):
    """يوسّع الصندوق المكتشف بهامش يضمن تغطية كامل جسم السيارة."""
    H, W = frame_shape[:2]
    x, y, w, h = box
    nx = int(max(0, x - w * PAD_LEFT))
    ny = int(max(0, y - h * PAD_TOP))
    nx2 = int(min(W, x + w + w * PAD_RIGHT))
    ny2 = int(min(H, y + h + h * PAD_BOTTOM))
    return nx, ny, nx2 - nx, ny2 - ny


def classify(s, v, h):
    """يحدد اسم اللون: يفرّق أولاً بين عديم اللون (أبيض/أسود/رمادي)
    والملوّن (يعتمد الصبغة Hue بغض النظر عن مدى الإضاءة/الظل)."""
    if s < 35:
        if v < 90:
            return "black"
        if v >= 165:
            return "white"
        return "gray"
    if h < 10 or h >= 170:
        return "red"
    if 10 <= h < 25:
        return "orange"
    if 25 <= h < 35:
        return "yellow"
    if 35 <= h < 85:
        return "green"
    if 85 <= h < 130:
        return "blue"
    if 130 <= h < 160:
        return "purple"
    return "pink"


def dominant_color(frame, box):
    """يحلل لون السيارة من مركز الصندوق فقط (يتجنب حواف قد تكون خلفية)،
    ويفرّق بين سيارة عديمة اللون وسيارة ملوّنة قبل التصنيف."""
    x, y, w, h = box
    cx0 = int(x + w * 0.25)
    cx1 = int(x + w * 0.75)
    cy0 = int(y + h * 0.25)
    cy1 = int(y + h * 0.75)
    roi = frame[cy0:cy1, cx0:cx1]
    if roi.size == 0:
        return "unknown"

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    h_ch = hsv[:, :, 0].ravel().astype(float)
    s_ch = hsv[:, :, 1].ravel().astype(float)
    v_ch = hsv[:, :, 2].ravel().astype(float)

    # استبعاد الظل الشديد والانعكاس الشديد (يشوهون التقدير)
    ok = (v_ch > 25) & (v_ch < 253)
    h_ch, s_ch, v_ch = h_ch[ok], s_ch[ok], v_ch[ok]
    if len(v_ch) == 0:
        return "unknown"

    low_sat_frac = (s_ch < 35).mean()

    if low_sat_frac > 0.55:
        # الأغلبية عديمة اللون (أبيض/رمادي/أسود): نصنّف حسب الإضاءة
        v_use = v_ch[s_ch < 35]
        med_v = np.percentile(v_use, 80)
        return classify(0, med_v, 0)

    # سيارة ملوّنة: نستخدم قمة هستوجرام الصبغة على البكسلات المشبعة فقط
    chroma = s_ch > 40
    hh = h_ch[chroma] if chroma.sum() >= 30 else h_ch
    hist, edges = np.histogram(hh, bins=30, range=(0, 180))
    peak_bin = np.argmax(hist)
    peak_hue = (edges[peak_bin] + edges[peak_bin + 1]) / 2
    med_s = np.median(s_ch[chroma]) if chroma.sum() > 0 else np.median(s_ch)
    med_v = np.median(v_ch[chroma]) if chroma.sum() > 0 else np.median(v_ch)
    return classify(med_s, med_v, peak_hue)


def main():
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("خطأ: تعذر فتح الفيديو. تأكد من صحة المسار:")
        print(video_path)
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps:
        fps = 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_area = frame_width * frame_height

    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    print("جاري معالجة الفيديو كاملاً... يرجى الانتظار.", flush=True)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        mask = compute_focus_mask(frame)
        seed_box = find_main_object(mask, frame_area)
        result = frame.copy()

        if seed_box is not None:
            x, y, w, h = pad_box(seed_box, frame.shape)
            color_key = dominant_color(frame, (x, y, w, h))

            # عزل الخلفية: تحويلها رمادي مع إبقاء السيارة كاملة ملونة
            box_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
            cv2.rectangle(box_mask, (x, y), (x + w, y + h), 255, -1)
            inv_mask = cv2.bitwise_not(box_mask)

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

            colored_part = cv2.bitwise_and(frame, frame, mask=box_mask)
            gray_part = cv2.bitwise_and(gray_bgr, gray_bgr, mask=inv_mask)
            result = cv2.add(colored_part, gray_part)

            cv2.rectangle(result, (x, y), (x + w, y + h), (0, 0, 255), 2)
            label = f"Car ({color_key})"
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            label_y = max(text_h + 12, y)
            cv2.rectangle(result, (x, label_y - text_h - 12), (x + text_w + 10, label_y), (0, 0, 255), -1)
            cv2.putText(result, label, (x + 5, label_y - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(result)
        frame_count += 1
        print(f"تمت معالجة {frame_count}/{total_frames} إطار...", end="\r", flush=True)

    cap.release()
    out.release()
    print(f"\nتم الانتهاء بنجاح! تم حفظ الفيديو المعالج في:\n{output_path}", flush=True)


if __name__ == "__main__":
    main()

جاري معالجة الفيديو كاملاً... يرجى الانتظار.
تمت معالجة 240/240 إطار...
تم الانتهاء بنجاح! تم حفظ الفيديو المعالج في:
D:\coop traning\weak3\processed_output.mp4
